<a href="https://colab.research.google.com/github/jrlt-opensource/Task-orchestration-using-Gemini-Robotics-ER-2/blob/main/Task_orchestration_using_Gemini_Robotics_ER_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook demonstrates how to use Google's VLM, Gemini Robotics ER 2 for Task Orchestration.**bold text**


Before running this notebook, please get your own API key from https://aistudio.google.com/api-keys . Create an API key, copy and paste it the variable, GEMINI_API_KEY, below.

DO NOT COMMIT YOUR API KEY TO A PUBLIC REPOSITORY!!!

In [ ]:
GEMINI_API_KEY = "Enter your API Key here. Get one from aistudio.google.com"

In [ ]:
def move(x, y, high):
  print(f"Mock Robot: Moving to coordinates: {x}, {y}, {'high above table' if high else 'down at table level'}")

def setGripperState(opened):
  print(f"Mock Robot: {'Opening gripper' if opened else 'Closing gripper'}")

robot_origin_y = 300
robot_origin_x = 500

move_function = {
    "type": "function",
    "name": "move",
    "description": "Moves the arm to the given coordinates.",
    "parameters": {
        "type": "object",
        "properties": {
            "x": {"type": "integer", "description": "X coordinate relative to the origin"},
            "y": {"type": "integer", "description": "Y coordinate relative to the origin"},
            "high": {"type": "boolean", "description": "Set to True to lift the robot arm above the scene for avoiding obstacles. Set to False to place the gripper on the surface."}
        },
        "required": ["x", "y", "high"]
    }
}

set_gripper_state_function = {
    "type": "function",
    "name": "setGripperState",
    "description": "Opens or closes the robot's gripper.",
    "parameters": {
        "type": "object",
        "properties": {
            "opened": {"type": "boolean", "description": "True opens the gripper, False closes the gripper."}
        },
        "required": ["opened"]
    }
}

In [ ]:
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/soarm-block.png -O soarm-block.png -q

In [ ]:
from PIL import Image
from io import BytesIO
import base64


image_path = "soarm-block.png"

img = Image.open(image_path)

buffered = BytesIO()
img.save(buffered, format="PNG")
img_b64 = base64.b64encode(buffered.getvalue()).decode()

In [ ]:
from IPython.display import display
display(img)

In [ ]:
import json
from google import genai

max_steps = 15 # Safety limit to prevent infinite loops
step_count = 0
MODEL_ID = "gemini-robotics-er-2-preview"

client = genai.Client(api_key=GEMINI_API_KEY)

def createInteraction(client, **kwargs):
    """Create an interaction using the provided API parameters."""
    return client.interactions.create(**kwargs)


def execute_function_call(step):
    """Execute a function requested by the model."""
    function_map = {
        "move": move,
        "setGripperState": setGripperState,
    }

    function = function_map.get(step.name)

    if function is None:
        print(f"Unknown function: {step.name}")
        result = {"status": "error", "message": "Unknown function"}
    else:
        function(**step.arguments)
        result = {"status": "success"}

    return {
        "type": "function_result",
        "name": step.name,
        "call_id": step.id,
        "result": [
            {"type": "text", "text": json.dumps(result)}
        ],
    }


def get_function_calls(interaction):
    """Extract function calls from an interaction."""
    return [
        step
        for step in interaction.steps
        if step.type == "function_call"
    ]


def run_agentic_loop(client, interaction, max_steps=15):
    """Execute function calls until the model finishes or the limit is reached."""
    for step_count in range(1, max_steps + 1):
        tool_calls = get_function_calls(interaction)

        if not tool_calls:
            print("Sequence complete.")
            if interaction.output_text:
                print(f"Model Summary: {interaction.output_text}")
            return interaction

        function_results = [
            execute_function_call(step)
            for step in tool_calls
        ]

        interaction = createInteraction(
            client,
            model=MODEL_ID,
            previous_interaction_id=interaction.id,
            tools=[move_function, set_gripper_state_function],
            input=function_results,
        )

    print(f"Safety limit reached after {max_steps} steps.")
    return interaction


# 1. Define the prompt (unchanged)
prompt = (
    "You are a robotic arm with six degrees-of-freedom. "
    f"The origin point for calculating the moves is at normalized point y={robot_origin_y}, x={robot_origin_x}. "
    "Use this as the new (0,0) for calculating moves, allowing x and y to be negative.\n\n"
    "Find the blue block and the orange bowl. Calculate their coordinates relative to the origin.\n"
    "Perform a pick and place operation where you pick up the blue block and place it into the orange bowl. "
    "Call the appropriate sequence of functions to complete this operation."
)


# 2. Create the initial interaction
interaction = createInteraction(
    client,
    model=MODEL_ID,
    input=[
        {
            "type": "user_input",
            "content": [
                {
                    "type": "image",
                    "data": img_b64,
                    "mime_type": "image/png",
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ],
    tools=[move_function, set_gripper_state_function],
    generation_config={"thinking_level": "low"},
)

print("\n--- Executing Orchestrated Plan ---")

# 3. Execute the agentic loop
interaction = run_agentic_loop(
    client,
    interaction,
    max_steps=15,
)